In [ ]:
%autoreload 2

In [ ]:
%reload_ext autoreload
import os, sys, random
import numpy as np
import pandas as pd
import seaborn as sns

from scipy.stats import zscore
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib import gridspec, rcParams
from datetime import datetime as dt, timedelta
import pingouin as pg
import scipy.stats as stats
from fish import Gafftopsail
sys.path.append(r'/Users/zichenhe/miniforge3/envs/naumann_lab/2ptank/')#(r'C://Users//Zichen//anaconda3//envs//2ptank//Lib//site-packages//')
import utils, barcode
path = ('C:/Data/Imaging/260425_overlap/fish8_2/')
rcParams['font.size'] = 4  

In [ ]:
fish = Gafftopsail(path, filelist = ['stimulus', 'imaging', 'alignment'], sequence = 5)
fish.stimulus_df.stim_name= [str(s) for s in fish.stimulus_df.stim_name]

__EACH STIM__

In [ ]:
#axis 0: trial; axis 1: neuron; axis 2: frame
#from stationary start to duration + 20
f_pertrial_dict = barcode.get_pertrial_f(fish)

__barcoding__

In [ ]:
barred_dotneurons = barcode.select_dotbarcode(fish, f_pertrial_dict, baseline_s = 7, response_s =10, perc_trial_threshold = 1)

In [ ]:
barred_dotneurons = barcode.sort_dotneurons(fish, f_pertrial_dict, barred_dotneurons)

In [ ]:
fig = barcode.plot_dotneurons(fish, f_pertrial_dict, barred_dotneurons)

In [ ]:
plt.savefig(fish.path + '//Graphs//dot_barcode.png')
plt.show()
plt.close()

__timing plot__

In [ ]:
grating_stim = None
stim_responses = barcode.get_dot_dsi(fish, f_pertrial_dict,baseline_s =7, response_s = 15, grating_stim = grating_stim)

In [ ]:
#combine barcoding and stim_responses
for stim, neurons in barred_dotneurons.items():
    stim_responses[f'barred_{stim}'] = stim_responses.index.isin(neurons)

#save it
os.makedirs(os.path.join(fish.path, "analyze_results"), exist_ok=True)
if grating_stim is not None:
    stim_responses.to_csv(fish.path + f'//analyze_results//{grating_stim}dot_responses.csv')
else:
    stim_responses.to_csv(fish.path + f'//analyze_results//dot_responses.csv')

In [ ]:
barcode.plot_timing(fish, stim_responses)
plt.savefig(fish.path + '//Graphs//dot_rainbow.png')
plt.show()
plt.close()

In [ ]:
#look at it across gratings
for grating_stim in ['forward', 'left', 'right', 'backward']:
    stim_responses = barcode.get_dot_dsi(fish, f_pertrial_dict, baseline_s=7, response_s=15, grating_stim=grating_stim)
    #combine barcoding and stim_responses
    for stim, neurons in barred_dotneurons.items():
        stim_responses[f'barred_{stim}'] = stim_responses.index.isin(neurons)

    #save it
    os.makedirs(os.path.join(fish.path, "analyze_results"), exist_ok=True)
    if grating_stim is not None:
        stim_responses.to_csv(fish.path + f'//analyze_results//dot_responses_[{grating_stim}].csv')
    else:
        stim_responses.to_csv(fish.path + f'//analyze_results//dot_responses.csv')
    barcode.plot_timing(fish, stim_responses)
    plt.savefig(fish.path + f'//Graphs//dot_rainbow_[{grating_stim}].png')
    plt.show()
    plt.close()

__get f__

In [ ]:
def get_f(fish, barred_dotneurons):
    """Get the avg trace of a population of cells"""
    #look at vis responsive cells
    region_list = [['prosencephalon_(forebrain)', 'mesencephalon_(midbrain)'], 'rhombencephalon_(hindbrain)', 'tectum', 'pretectum']

    #get traces
    f_dict = {'forebrain':{}, 'hindbrain':{}, 'tectum': {}, 'pretectum': {}}

    fig, ax = plt.subplots(len(barred_dotneurons), 1 + 2 * len(region_list), figsize = (20, 10))
    #each direction: line vs dot line
    for s, stim in enumerate(barred_dotneurons):
        stim_n = set(barred_dotneurons[stim])
        ax_imshow = ax[s, 0]
        ax_imshow.imshow(fish.img_dict[2], origin = 'lower')
        ax_imshow.set_axis_off()
        for r, (region, region_c) in enumerate(zip(region_list, ['white', 'grey', 'yellow','green'])):
            ax_line = ax[s, r * 2 + 1]
            ax_heat = ax[s, r * 2 + 2]
            f_df = []
            if type(region) == list:
                region_n = set(fish.apos_all.index[fish.apos_all['regions'].apply(lambda x: any(ri in x for ri in region))])#set(fish.apos_all.index)#
            else:
                region_n = set(fish.apos_all.index[fish.apos_all['regions'].apply(lambda x: region in x)])
            region_n = list(region_n & stim_n)

            for g, (grating, color) in enumerate(zip([None, 'forward', 'left', 'right', 'backward'], [utils.pink, utils.omr_colors['forward'], utils.omr_colors['left'], utils.omr_colors['right'], utils.omr_colors['backward']])):
                if grating == None:
                    real_stim = stim
                else:
                    real_stim =  str([stim, grating])

                #get fluorscnece
                f_acrosstrial = f_pertrial_dict[real_stim][:, region_n].mean(axis = 0)
                f_mean = f_acrosstrial.mean(axis = 0)
                f_sem = f_acrosstrial.std(axis=0) / np.sqrt(f_acrosstrial.shape[0])
                df = pd.DataFrame(f_acrosstrial)
                if len(df) > 0:
                    df.loc[:, 'barred'] = stim
                    df.loc[:, 'stim'] = real_stim
                f_df.append(df)
                #start plotting
                ax_imshow.scatter(fish.pos_all.loc[region_n, 'xpos'], fish.pos_all.loc[region_n, 'ypos'], marker = ',', s = 1, color = region_c)

                ax_line.plot(f_mean, c=color)
                ax_line.fill_between(np.arange(len(f_mean)),f_mean - f_sem, f_mean + f_sem,color=color,alpha=0.3, linewidth = 0)
                ax_line.set_ylim([0.1, 0.6])

                ax_heat.imshow(f_acrosstrial, aspect='auto',extent=[0, len(f_mean), g, g+1],cmap='viridis', vmin = 0.2, vmax = 0.6)
                ax_heat.axhline(g, linestyle='--', color='white')
                ax_heat.set_ylim([0, g + 1])
            if s == 0:
                f_dict[list(f_dict.keys())[r]] = pd.concat(f_df, axis = 0, ignore_index = True)
            else:
                f_dict[list(f_dict.keys())[r]] = pd.concat([f_dict[list(f_dict.keys())[r]], pd.concat(f_df, axis = 0, ignore_index = True)], axis = 0, ignore_index = True)
    return f_dict, fig

In [ ]:
f_dict, fig = get_f(fish, barred_dotneurons)
plt.show()
plt.close()

In [ ]:
#combine across fish
paths =[
 r"C:\Data\Imaging\260415_overlap\fish3",r"C:\Data\Imaging\260415_overlap\fish3_2",
r"C:\Data\Imaging\260415_overlap\fish4", r"C:\Data\Imaging\260415_overlap\fish4_2",
     r"C:\Data\Imaging\260415_overlap\fish6", r"C:\Data\Imaging\260415_overlap\fish6_2",
r"C:\Data\Imaging\260425_overlap\fish1", r"C:\Data\Imaging\260425_overlap\fish1_2",
r"C:\Data\Imaging\260425_overlap\fish2",r"C:\Data\Imaging\260425_overlap\fish2_2",
r"C:\Data\Imaging\260425_overlap\fish3",r"C:\Data\Imaging\260425_overlap\fish3_2",
r"C:\Data\Imaging\260425_overlap\fish5",r"C:\Data\Imaging\260425_overlap\fish5_2",
 r"C:\Data\Imaging\260425_overlap\fish6", r"C:\Data\Imaging\260425_overlap\fish6_2",
r"C:\Data\Imaging\260425_overlap\fish7",r"C:\Data\Imaging\260425_overlap\fish7_2",
r"C:\Data\Imaging\260425_overlap\fish8",r"C:\Data\Imaging\260425_overlap\fish8_2",
r"C:\Data\Imaging\260425_overlap\fish9", r"C:\Data\Imaging\260425_overlap\fish9_2",
 r"C:\Data\Imaging\260425_overlap\fish10", r"C:\Data\Imaging\260425_overlap\fish10_2",
r"C:\Data\Imaging\260425_overlap\fish11", r"C:\Data\Imaging\260425_overlap\fish11_2",]
for path in paths:
    fish = Gafftopsail(path + '//', filelist=['stimulus', 'imaging', 'alignment'], sequence=5)
    fish.stimulus_df.stim_name = [str(s) for s in fish.stimulus_df.stim_name]
    #axis 0: trial; axis 1: neuron; axis 2: frame
    #from stationary start to duration + 20
    f_pertrial_dict = barcode.get_pertrial_f(fish)
    barred_dotneurons = barcode.select_dotbarcode(fish, f_pertrial_dict, baseline_s=7, response_s=10,
                                                  perc_trial_threshold=1)
    f_dict, fig = get_f(fish, barred_dotneurons)
    for region in ['forebrain', 'hindbrain', 'tectum', 'pretectum']:
            df = f_dict[region]
            df.to_csv(fish.path + f"//analyze_results//dot_{region}neurons.csv")
    plt.show()
    plt.close()